
---
# Curso: Data Mining - Semana 3

Docente: Soledad Espezúa (s.espezual@up.edu.pe)

---

<a name='manejoarchivos'></a>
# <font color=blue> Integración final, organización de datos y primeras visualizaciones</font>

**Objetivo:** unir en un solo flujo lo trabajado en los laboratorios previos: inspección, organización básica, validación de claves, integración de fuentes y primeras visualizaciones interactivas.

In [ ]:
import pandas as pd
import plotly.express as px

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 80)


# <font color=green> Parte 1. Cargar e inspeccionar las fuentes</font>
Antes de integrar, debemos entender qué contiene cada fuente.

In [ ]:
estudiantes = pd.read_csv("estudiantes.csv")
rendimiento = pd.read_csv("rendimiento.csv")

print("Fuente 1: estudiantes")
display(estudiantes.head())

print("Fuente 2: rendimiento")
rendimiento.head()


In [ ]:
print("Dimensiones")
print("estudiantes:", estudiantes.shape)
print("rendimiento:", rendimiento.shape)

print("\nColumnas de estudiantes:")
print(estudiantes.columns.tolist()) # Verificar nombres legibles y sin espacios

print("\nColumnas de rendimiento:")
rendimiento.columns.tolist()

In [ ]:
print("Tipos detectados por Pandas en estudiantes:")
display(estudiantes.dtypes) # Verificar adecuación del tipo-variable

print("Tipos detectados por Pandas en rendimiento:")
rendimiento.dtypes

###<font color=279CF5> 1.1 Diagnóstico rápido de valores faltantes</font>

No vamos a hacer una limpieza profunda todavía. Primero debemos detectar y registrar el problema.


In [ ]:
print("Valores faltantes en estudiantes:")
display(estudiantes.isna().sum())

print("Valores faltantes en rendimiento:")
rendimiento.isna().sum()

###<font color=279CF5> 1.2 Valores distintos en variables categóricas</font>

Esto ayuda a detectar categorías escritas de varias maneras, por ejemplo: `Sí`, `Si`, `SI`.


In [ ]:
print("Valores de internet_casa:")
display(estudiantes["internet_casa"].value_counts(dropna=False))

print("Valores de sexo:")
display(estudiantes["sexo"].value_counts(dropna=False))

print("Valores de resultado:")
rendimiento["resultado"].value_counts(dropna=False)

# <font color=green> Parte 2. Organizar datos antes de integrar</font>

A veces necesitamos resumir o agrupar información antes de construir una base de análisis.


In [ ]:
# Ejemplo: cantidad de estudiantes por tipo de colegio
conteo_colegio = estudiantes.groupby("tipo_colegio").size().reset_index(name="cantidad")
display(conteo_colegio)


In [ ]:
# Ejemplo: promedio de nota por resultado académico
resumen_resultado = rendimiento.groupby("resultado")[["nota_final", "asistencia_pct", "horas_estudio_semana"]].mean().round(2)
display(resumen_resultado)


# <font color=green>  Parte 3. `concat` versus `merge`</font>

Ambas operaciones combinan datos, pero no hacen lo mismo.

- `concat` sirve para **apilar** tablas con estructura similar.
- `merge` sirve para **emparejar** tablas usando una clave común.


###<font color=279CF5>  3.1 Cuándo usar `concat`</font>

Si tengo dos archivos con las mismas columnas y quiero unir sus filas.


In [ ]:
rendimiento_a = pd.read_csv("rendimiento_grupo_a.csv")
rendimiento_b = pd.read_csv("rendimiento_grupo_b.csv")

print("Grupo A:", rendimiento_a.shape)
print("Grupo B:", rendimiento_b.shape)

#Se apila los dos grupos uno debajo del otro (por defecto en el eje de filas axis=0)
#ignore_index=True: descarta índices originales y crea uno nuevo
rendimiento_concatenado = pd.concat([rendimiento_a, rendimiento_b], ignore_index=True)

print("Concatenado:", rendimiento_concatenado.shape)
rendimiento_concatenado.head()

###<font color=279CF5>   3.2 Cuándo usar `merge`</font>

Usamos merge cuando tenemos dos tablas con información complementaria sobre una misma unidad de análisis y queremos unirlas usando una clave común.

En este laboratorio:

* estudiantes contiene datos personales y de contexto.
* rendimiento contiene datos académicos.
* Ambas fuentes tienen la clave **id_estudiante**.
* La unidad de análisis esperada es el estudiante.

Por eso usamos `merge`: queremos construir una base integrada donde cada fila represente a un estudiante con sus datos personales y de rendimiento.

In [ ]:
# En nuestro caso, la clave propuesta es id_estudiante.
# Queremos emparejar datos personales con datos de rendimiento.
base = estudiantes.merge(
    rendimiento,
    on="id_estudiante", #Indica la clave común para unir las tablas
    how="inner"         # Conserva solo los registros que aparecen en ambas fuentes
)

base.head()

# <font color=green> Parte 4. Validar claves antes de integrar</font>

Antes de usar `merge`, revisamos si la clave aparece duplicada.

Esta revisión depende de la unidad de análisis. Si la unidad es **estudiante**, esperaríamos un registro por estudiante.


In [ ]:
#keep=False: sirve para marcar como duplicadas todas las filas que tienen un id_estudiante repetido, incluyendo la primera aparición.

print("Duplicados en estudiantes:")
duplicados_estudiantes = estudiantes[estudiantes.duplicated(subset=["id_estudiante"], keep=False)].sort_values("id_estudiante")
display(duplicados_estudiantes)

print("Duplicados en rendimiento:")
duplicados_rendimiento = rendimiento[rendimiento.duplicated(subset=["id_estudiante"], keep=False)].sort_values("id_estudiante")
display(duplicados_rendimiento)


# <font color=green> Parte 5. Integración auditada con `indicator=True`</font>

* Usamos `how="outer"` para observar todos los casos: los que cruzan y los que no cruzan.
* Usamos `indicator=True` para que merge() agregue una columna extra llamada **_merge**, que indica de dónde viene cada fila después de la integración.

In [ ]:
base_validacion = estudiantes.merge(
    rendimiento,
    on="id_estudiante",
    how="outer", #Conserva todos los registros de ambas tablas
    indicator=True,
    suffixes=("_est", "_rend") # solo tendrá efecto si las dos tablas tienen columnas con el mismo nombre
)

base_validacion

In [ ]:
conteo_cruce = base_validacion["_merge"].value_counts().reset_index() #cuenta cuántas filas hay de cada tipo de merge
conteo_cruce.columns = ["resultado_cruce", "cantidad"]

conteo_cruce

Eso significa:

* 12 registros cruzaron correctamente;
* 3 estudiantes están en la fuente estudiantes, pero no tienen registro en rendimiento;
* 2 registros están en rendimiento, pero no aparecen en la fuente estudiantes.

In [ ]:
fig = px.bar(
    conteo_cruce,
    x="resultado_cruce",
    y="cantidad",
    text="cantidad",
    title="Resultado de la integración de fuentes"
)

fig.show()


In [ ]:
print("Registros que no cruzan entre fuentes:")

no_cruzan = base_validacion[base_validacion["_merge"] != "both"]

columnas_mostrar = ["id_estudiante", "nombre", "nota_final", "_merge"]
no_cruzan[columnas_mostrar]

# <font color=green>Parte 6. Construir una base integrada para análisis</font>

Para simplificar el laboratorio, construiremos una base a nivel de estudiante.  
Conservaremos la primera aparición de cada `id_estudiante`.

En un proyecto real, esta decisión debe justificarse.


* keep="first": Conserva la primera aparición del id_estudiante y elimina las siguientes.
* keep="last": Conserva la última aparición del id_estudiante y elimina las anteriores.
* keep=False: Elimina todas las filas que tienen un duplicado.

In [ ]:
estudiantes_unicos = estudiantes.drop_duplicates(subset=["id_estudiante"], keep="first")
rendimiento_unico = rendimiento.drop_duplicates(subset=["id_estudiante"], keep="first")

base = estudiantes_unicos.merge(
    rendimiento_unico,
    on="id_estudiante",
    how="inner"
)

print("Tamaño de la base integrada para análisis:", base.shape)
base.head()

###<font color=279CF5> 6.1 Estandarizar las variables y categorías mínimas</font>

No haremos una limpieza completa. Solo corregimos categorías necesarias para graficar mejor.


In [ ]:
base["internet_casa_std"] = ( #crear internet_casa_std para comparar la salida
    base["internet_casa"]
    .astype(str) # asegura que todo sea texto
    .str.strip() # quita espacios al inicio y al final
    .str.lower() # pasa todo a minúsculas
    .replace({   # estandariza categorías
        "sí": "Sí",
        "si": "Sí",
        "no": "No",
        "": "No registrado",
        "nan": "No registrado"
    })
)

base["sexo_std"] = (
    base["sexo"]
    .astype(str)
    .str.strip()
    .str.lower()
    .replace({
        "f": "Femenino",
        "femenino": "Femenino",
        "m": "Masculino",
        "masculino": "Masculino",
        "masc": "Masculino"
    })
)

base[["id_estudiante", "sexo", "sexo_std", "internet_casa", "internet_casa_std"]]

# <font color=green>Parte 7. Primeras visualizaciones con Plotly</font>

Estas visualizaciones no reemplazan el EDA formal de la próxima semana.  
Hoy las usamos para generar primeras lecturas útiles para el hito del proyecto.


###<font color=279CF5> 7.1 Barras: estudiantes por tipo de colegio</font>

In [ ]:
conteo_colegio = base["tipo_colegio"].value_counts().reset_index()
conteo_colegio.columns = ["tipo_colegio", "cantidad"]

fig = px.bar(
    conteo_colegio,
    x="tipo_colegio",
    y="cantidad",
    text="cantidad",
    title="Estudiantes por tipo de colegio"
)

fig.show()

###<font color=279CF5> 7.2 Histograma: distribución de la nota final</font>

In [ ]:
fig = px.histogram(
    base,
    x="nota_final",
    nbins=8,
    title="Distribución de la nota final",
    marginal="box" # Ademas muestra un boxplot de la varaible
)

fig.show()


###<font color=279CF5> 7.3 Boxplot: nota final del estudiante según internet en casa</font>

In [ ]:
fig = px.box(
    base,
    x="internet_casa_std",
    y="nota_final",
    points="all", # Ademas muestra cada dato individual como un punto
    title="Nota final según disponibilidad de internet en casa"
)

fig.show()

###<font color=279CF5> 7.4 Dispersión: horas de estudio y nota final</font>

In [ ]:
fig = px.scatter(
    base,                     # DataFrame que contiene la base integrada
    x="horas_estudio_semana",     # Variable del eje X: horas de estudio semanal
    y="nota_final",               # Variable del eje Y: nota final obtenida

    color="resultado",      # Colorea los puntos según el resultado académico
                            # por ejemplo: Aprobado o Desaprobado

    size="asistencia_pct",  # El tamaño del punto representa el porcentaje de asistencia.
                            # A mayor asistencia, mayor tamaño del punto.

    hover_data=[
        "id_estudiante",     # Muestra el ID del estudiante al pasar el cursor
        "tipo_colegio",      # Muestra el tipo de colegio
        "internet_casa_std"  # Muestra si tiene internet en casa, ya estandarizado
    ],

    title="Horas de estudio vs. nota final"  # Título del gráfico
)


fig.show()


Cambiar los colores de Aprobado y Desaprobado
```
    color_discrete_map={
        "Aprobado": "green",
        "Desaprobado": "red"
    }
```



# <font color=green> Parte 8. Del gráfico al hallazgo</font>

Un gráfico debe convertirse en una interpretación breve y prudente.


In [ ]:
hallazgos = pd.DataFrame({
    "grafico": [
        "Resultado de integración",
        "Distribución de nota final",
        "Nota según internet en casa",
        "Horas de estudio vs. nota final"
    ],
    "observacion": [
        "",
        "",
        "",
        ""
    ],
    "posible_hallazgo_o_hipotesis": [
        "",
        "",
        "",
        ""
    ],
    "precaucion": [
        "Revisar registros que no cruzan antes de excluirlos.",
        "Una distribución no prueba causas.",
        "Puede haber otros factores asociados.",
        "Relación visual no implica causalidad."
    ]
})

hallazgos


## Actividad final para grupos

Cada grupo debe aplicar el mismo flujo a su proyecto.

### Entrega:

1. Cargar al menos dos fuentes.
2. Definir la unidad de análisis.
3. Identificar la clave o claves de unión.
4. Validar duplicados en las claves.
5. Integrar con `indicator=True`.
6. Mostrar cuántos registros cruzan y cuántos no cruzan.
7. Construir una base integrada para análisis.
8. Crear al menos **3 gráficos iniciales con Plotly**.
9. Redactar **3 hallazgos preliminares**.

### Presentar en clase

- Base integrada.
- Tabla de validación de claves.
- Tres gráficos.
- Tres hallazgos prudentes y justificados.


In [ ]:
# Opcional: exportar la base integrada para continuar la próxima semana
base.to_csv("base_integrada_semana3.csv", index=False, encoding="utf-8-sig")

print("Archivo generado: base_integrada_semana3.csv")

`encoding="utf-8-sig"` lo usamos para que programas como Excel reconozcan correctamente los caracteres especiales en español, por ejemplo: á, é, í, ó, ú, ñ, Sí, Perú, Matemática